In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.decomposition import PCA
from umap import UMAP

In [2]:
embedings_cache_path = "./embed_cache.pkl"
#load from pickle
import pickle
with open(embedings_cache_path, "rb") as f:
    embeddings_cache = pickle.load(f)

#loaded embeddings cache is a dict of text to embedding
print(f"Loaded embeddings cache with {len(embeddings_cache)} entries")

Loaded embeddings cache with 43047 entries


In [47]:
#load answers text cache
answers_text_cache_path = "./data/r2.csv"
df_answers = pd.read_csv(answers_text_cache_path, keep_default_na=False)
df_answers = df_answers[df_answers["REPETITION"] == 1]  # Filter out empty answers
df_answers.info()

<class 'pandas.DataFrame'>
Index: 10000 entries, 0 to 85980
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   AGENT         10000 non-null  str  
 1   VIDEO         10000 non-null  str  
 2   BLOCK         10000 non-null  int64
 3   QUESTION_NUM  10000 non-null  int64
 4   REPETITION    10000 non-null  int64
 5   ANSWER        10000 non-null  str  
dtypes: int64(3), str(3)
memory usage: 1.5 MB


In [4]:
#first reduce all embedding cache using pca to 2 dimensions and store then into a df with agent video question answer and embedding
embeddings = np.stack(df_answers["ANSWER"].map(embeddings_cache))
print(f"Embed matrix shape: {embeddings.shape}")

Embed matrix shape: (10000, 384)


In [5]:
coords_PCA = np.zeros((embeddings.shape[0], 2))
coords_UMAP = np.zeros((embeddings.shape[0], 2))
for block in [1, 2, 3, 4]:
    mask = df_answers["BLOCK"] == block
    print(f"Processing block {block} with {mask.sum()} entries")
    if mask.any():
        pca_block = PCA(n_components=2)
        umap_block = UMAP(n_components=2,metric="cosine",local_connectivity = 1, n_jobs=16, random_state=42)
        coords_PCA[mask.values] = pca_block.fit_transform(embeddings[mask.values])
        coords_UMAP[mask.values] = umap_block.fit_transform(embeddings[mask.values])


Processing block 1 with 2500 entries


/home/andre/miniconda3/envs/torch/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Processing block 2 with 2500 entries


/home/andre/miniconda3/envs/torch/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Processing block 3 with 2500 entries


/home/andre/miniconda3/envs/torch/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Processing block 4 with 2500 entries


/home/andre/miniconda3/envs/torch/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
interactive = df_answers[["AGENT", "VIDEO","BLOCK", "QUESTION_NUM", "ANSWER"]].assign(pca_X=coords_PCA[:, 0], pca_Y=coords_PCA[:, 1])
interactive = interactive.assign(umap_X=coords_UMAP[:, 0], umap_Y=coords_UMAP[:, 1])

# --- Merge vectorizado ---
def get_group_color(agent):
    if "human" in agent:
        if "nyc" in agent:
            return "nyc"
        else:
            return "lima"
    else:
        return "vlm"
    
color_map = {
    "nyc": "#0000ff",
    "lima": "#ff0000",
    "vlm": "#00ff00",
}

interactive["group"] = interactive["AGENT"].map(get_group_color)

In [45]:
print(interactive.head())

          AGENT         VIDEO  BLOCK  QUESTION_NUM  \
0  human_lima_1  Robusto2_153      1             1   
1  human_lima_2  Robusto2_153      1             1   
2  human_lima_3  Robusto2_153      1             1   
3  human_lima_4  Robusto2_153      1             1   
4  human_lima_5  Robusto2_153      1             1   

                                              ANSWER     pca_X     pca_Y  \
0  The ego vehicle is accelerating slowly because...  0.418389  0.064279   
1            The ego vehicle is turning to the right  0.522257  0.002757   
2  the ego vehicle brakes and steers slightly to ...  0.454121 -0.045594   
3                                   Braking to yield  0.047026 -0.055908   
4  The ego vehicle is moving forward while mainta...  0.507037 -0.105782   

     umap_X     umap_Y group  
0 -4.911002  14.774331  lima  
1 -5.882582  12.445625  lima  
2 -8.486742  18.322048  lima  
3 -4.129244   8.836326  lima  
4 -6.834734  13.383281  lima  


In [63]:
import jscatter
import pandas as pd
import numpy as np

BLOCK_TO_PLOT = int(input("Enter block number to plot (1-4): "))
df_block = (
    interactive[interactive["BLOCK"] == BLOCK_TO_PLOT]
    .copy()
)

# Convertir columnas del tooltip a string puro    
df_answers["AGENT"] = df_answers["AGENT"].astype(str)

# Sample data
# Create an interactive scatter plot
scatter = jscatter.Scatter(
    data= df_block,  # Filter for block 1
    x='umap_X',
    y='umap_Y',
    color_by='group',
    color_map=color_map,
    opacity=0.7,
    width=800,
    height=600,   
)

scatter.axes(grid=True)
scatter.axes(labels=['PCA 1', 'PCA 2'])
scatter.tooltip(
  enable=True,
  properties=['AGENT', 'VIDEO', 'QUESTION_NUM', 'ANSWER']
)
scatter.show()